# Feature Engineering

Handling missing data: fill with median

Random state: 42

## Imports and loading data

In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import pickle
import warnings
warnings.filterwarnings('ignore')
 
print("Imports successful")
 
# Load the EDA output
df = pd.read_csv('../EDA/mlb_pitches_eda.csv')
print(f"Loaded {len(df)} pitches")

Imports successful
Loaded 600414 pitches


## Features 1 and 2: Handedness

Reading pitcher handedness and batter handedness. Left-Left/Right-Right favors the pitcher, Left-Right/Right-Left favors the batter.

In [4]:
# Pitcher handedness: p_throws is L or R
df['pitcher_hand'] = df['p_throws'].map({'L': 0, 'R': 1})
print(f"Pitcher handedness distribution:")
print(df['p_throws'].value_counts())
print(f"  - Encoded as: L=0, R=1")
 
# Batter handedness: stand is L, R, or S (switch)
df['batter_hand'] = df['stand'].map({'L': 0, 'R': 1, 'S': 0.5})
print(f"\nBatter handedness distribution:")
print(df['stand'].value_counts())
print(f"  - Encoded as: L=0, R=1, S=0.5")
 
# Handedness matchup (4 categories)
# TODO: Data analysis on whether switch hitters always choose the opposite side of the pitcher to a significant degree. 
# If so, we can treat switch hitters as the opposite side of the pitcher.
def encode_matchup(row):
    p = row['p_throws']
    b = row['stand']
    matchups = {
        ('L', 'L'): 0,  # Left on Left
        ('L', 'R'): 1,  # Left on Right
        ('L', 'S'): 2,  # Left on Switch
        ('R', 'L'): 3,  # Right on Left
        ('R', 'R'): 4,  # Right on Right
        ('R', 'S'): 5,  # Right on Switch
    }
    return matchups.get((p, b), np.nan)
 
df['handedness_matchup'] = df.apply(encode_matchup, axis=1)
print(f"\nHandedness matchup distribution:")
print(df['handedness_matchup'].value_counts().sort_index())

Pitcher handedness distribution:
p_throws
R    423084
L    177330
Name: count, dtype: int64
  - Encoded as: L=0, R=1

Batter handedness distribution:
stand
R    305128
L    295286
Name: count, dtype: int64
  - Encoded as: L=0, R=1, S=0.5

Handedness matchup distribution:
handedness_matchup
0     59522
1    117808
3    235764
4    187320
Name: count, dtype: int64


## Feature 3: Count

In [5]:
# Already encoded in EDA as count_state; split back into separate features
df['balls'] = df['balls'].fillna(0).astype(int)
df['strikes'] = df['strikes'].fillna(0).astype(int)
 
print(f"Balls distribution:")
print(df['balls'].value_counts().sort_index())
print(f"\nStrikes distribution:")
print(df['strikes'].value_counts().sort_index())

Balls distribution:
balls
0    272220
1    177630
2    100762
3     49802
Name: count, dtype: int64

Strikes distribution:
strikes
0    239982
1    181664
2    178768
Name: count, dtype: int64


## Features 4 and 5: Outs and Inning

In [6]:
df['outs_when_up'] = df['outs_when_up'].fillna(0).astype(int)
df['inning'] = df['inning'].fillna(1).astype(int)
 
print(f"Outs distribution:")
print(df['outs_when_up'].value_counts().sort_index())
print(f"\nInning distribution (first 10 innings):")
print(df[df['inning'] <= 10]['inning'].value_counts().sort_index())

Outs distribution:
outs_when_up
0    206147
1    198832
2    195435
Name: count, dtype: int64

Inning distribution (first 10 innings):
inning
1     69524
2     66968
3     67159
4     66855
5     66960
6     67673
7     68156
8     68807
9     51480
10     5092
Name: count, dtype: int64


## Feature 6: Pitcher Arsenal

In [7]:
# For each pitcher, calculate % of each pitch type
pitcher_pitch_counts = df.groupby(['pitcher', 'pitch_type']).size().unstack(fill_value=0)
pitcher_pitch_totals = pitcher_pitch_counts.sum(axis=1)
pitcher_arsenal = pitcher_pitch_counts.div(pitcher_pitch_totals, axis=0) * 100
 
print(f"\nPitcher arsenal (first 5 pitchers, top 5 pitch types):")
print(pitcher_arsenal.iloc[:5, :5])

# Join back to main dataframe (each row gets their pitcher's arsenal)
df = df.merge(pitcher_arsenal, left_on='pitcher', right_index=True, how='left')
 
# Rename columns for clarity (FF_pct, SL_pct, etc.)
pitch_types = pitcher_arsenal.columns.tolist()
arsenal_cols = {pitch: f'{pitch}_pct' for pitch in pitch_types}
df = df.rename(columns=arsenal_cols)
 
print(f"\nAdded {len(arsenal_cols)} arsenal columns")
print(f"  - Example columns: {list(arsenal_cols.values())[:5]}")


Pitcher arsenal (first 5 pitchers, top 5 pitch types):
pitch_type         CH   CS         CU   EP   FA
pitcher                                        
434378      10.000000  0.0  17.500000  0.0  0.0
445276       0.000000  0.0   0.000000  0.0  0.0
453286      16.000000  0.0  13.837838  0.0  0.0
455119       0.000000  0.0   5.645161  0.0  0.0
471911      30.894309  0.0   0.000000  0.0  0.0

Added 17 arsenal columns
  - Example columns: ['CH_pct', 'CS_pct', 'CU_pct', 'EP_pct', 'FA_pct']


## Feature 7: Batter's Career Stats

Focusing on K% and Contact%. Could introduce Whiff% in the future too?

In [8]:
# For each batter, calculate:
# - Strikeout rate: how often they strike out
# - Contact rate: how often they put the ball in play
 
# Count strikeouts: where 'events' contains 'strikeout'
batter_strikeouts = df[df['events'].str.contains('strikeout', case=False, na=False)].groupby('batter').size()
batter_at_bats = df.groupby('batter').size()  # Total pitches as proxy for at-bats (roughly)
 
batter_k_rate = (batter_strikeouts / batter_at_bats * 100).fillna(0)
 
print(f"Batter K% stats (first 10 batters):")
print(batter_k_rate.head(10))
print(f"\nOverall K% distribution:")
print(f"  Mean: {batter_k_rate.mean():.1f}%")
print(f"  Median: {batter_k_rate.median():.1f}%")
print(f"  Std: {batter_k_rate.std():.1f}%")
 
# Join back to main dataframe
df = df.merge(batter_k_rate.rename('batter_k_rate'), left_on='batter', right_index=True, how='left')
df['batter_k_rate'] = df['batter_k_rate'].fillna(df['batter_k_rate'].median())
 
print(f"\nAdded batter_k_rate")

Batter K% stats (first 10 batters):
batter
457705     6.179775
467793     7.547170
500743     2.896552
502054    11.666667
502671     5.973451
506702    10.112360
514888     5.618649
516782     6.336940
518595     7.258065
518692     4.013811
dtype: float64

Overall K% distribution:
  Mean: 6.3%
  Median: 6.0%
  Std: 2.7%

Added batter_k_rate


## Feature 8: Pitcher Workload Bucket

In [9]:
# Convert workload_bucket to numeric
workload_map = {'0-20': 0, '21-50': 1, '51-80': 2, '81+': 3}
df['workload_bucket_encoded'] = df['workload_bucket'].map(workload_map)
 
print(f"Workload bucket distribution:")
print(df['workload_bucket'].value_counts().sort_index())

Workload bucket distribution:
workload_bucket
0-20     286060
21-50    163438
51-80    113039
81+       37877
Name: count, dtype: int64


## Feature 9: Target variable: Pitch Type

In [10]:
# Encode pitch types as integers
pitch_type_encoder = LabelEncoder()
df['pitch_type_encoded'] = pitch_type_encoder.fit_transform(df['pitch_type'])
 
print(f"Pitch types and encodings:")
for i, pitch in enumerate(pitch_type_encoder.classes_):
    count = (df['pitch_type'] == pitch).sum()
    pct = count / len(df) * 100
    print(f"  {i:2d}: {pitch:3s} ({count:6d} pitches, {pct:5.1f}%)")
 
# Save encoder for later
with open('pitch_type_encoder.pkl', 'wb') as f:
    pickle.dump(pitch_type_encoder, f)
print(f"\nSaved pitch_type_encoder to pickle")

Pitch types and encodings:
   0: CH  ( 67302 pitches,  11.2%)
   1: CS  (   122 pitches,   0.0%)
   2: CU  ( 38290 pitches,   6.4%)
   3: EP  (  1016 pitches,   0.2%)
   4: FA  (   648 pitches,   0.1%)
   5: FC  ( 47752 pitches,   8.0%)
   6: FF  (183636 pitches,  30.6%)
   7: FO  (   401 pitches,   0.1%)
   8: FS  ( 19437 pitches,   3.2%)
   9: KC  (  9600 pitches,   1.6%)
  10: KN  (   247 pitches,   0.0%)
  11: PO  (    34 pitches,   0.0%)
  12: SI  ( 99870 pitches,  16.6%)
  13: SL  ( 79376 pitches,  13.2%)
  14: ST  ( 50012 pitches,   8.3%)
  15: SV  (  2660 pitches,   0.4%)
  16: UN  (    11 pitches,   0.0%)

Saved pitch_type_encoder to pickle


## Selecting Features for XGBoost

In [11]:
# List of all feature columns
feature_cols = [
    'balls',
    'strikes',
    'pitcher_hand',
    'batter_hand',
    'handedness_matchup',
    'outs_when_up',
    'inning',
    'workload_bucket_encoded',
    'batter_k_rate',
]
 
# Add arsenal columns (FF_pct, SL_pct, etc.)
feature_cols.extend([col for col in df.columns if col.endswith('_pct')])
 
print(f"XGBoost features ({len(feature_cols)} total):")
for i, col in enumerate(feature_cols):
    print(f"  {i+1:2d}. {col}")
 
# Create feature matrix and target
X = df[feature_cols].copy()
y = df['pitch_type_encoded'].copy()
 
# Check for missing values
print(f"\nMissing values per feature:")
missing = X.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0])
    # Fill any remaining NaNs with median
    X = X.fillna(X.median())
    print("Filled with median values")
else:
    print("  None!")
 
print(f"\nFeature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")

XGBoost features (26 total):
   1. balls
   2. strikes
   3. pitcher_hand
   4. batter_hand
   5. handedness_matchup
   6. outs_when_up
   7. inning
   8. workload_bucket_encoded
   9. batter_k_rate
  10. CH_pct
  11. CS_pct
  12. CU_pct
  13. EP_pct
  14. FA_pct
  15. FC_pct
  16. FF_pct
  17. FO_pct
  18. FS_pct
  19. KC_pct
  20. KN_pct
  21. PO_pct
  22. SI_pct
  23. SL_pct
  24. ST_pct
  25. SV_pct
  26. UN_pct

Missing values per feature:
  None!

Feature matrix shape: (600414, 26)
Target shape: (600414,)


## Train/Test Split

IMPORTANT: Holdout by pitcher

In [12]:
# Get unique pitchers
unique_pitchers = df['pitcher'].unique()
print(f"Total unique pitchers: {len(unique_pitchers)}")
 
# Split pitchers into train/test (80/20 by pitcher)
np.random.seed(42)
train_pitchers = np.random.choice(unique_pitchers, size=int(0.8 * len(unique_pitchers)), replace=False)
test_pitchers = np.setdiff1d(unique_pitchers, train_pitchers)
 
print(f"Train pitchers: {len(train_pitchers)}")
print(f"Test pitchers: {len(test_pitchers)}")
 
# Create train/test indices
train_idx = df['pitcher'].isin(train_pitchers)
test_idx = df['pitcher'].isin(test_pitchers)
 
X_train = X[train_idx].copy()
y_train = y[train_idx].copy()
X_test = X[test_idx].copy()
y_test = y[test_idx].copy()
 
print(f"\nTrain set: {len(X_train)} pitches from {train_idx.sum()} rows")
print(f"Test set: {len(X_test)} pitches from {test_idx.sum()} rows")
 
# Verify no pitcher leakage
train_pitcher_set = set(df[train_idx]['pitcher'].unique())
test_pitcher_set = set(df[test_idx]['pitcher'].unique())
print(f"\nPitcher leakage check: {len(train_pitcher_set & test_pitcher_set)} overlapping pitchers")
if len(train_pitcher_set & test_pitcher_set) == 0:
    print("No pitcher leakage. Train and test are isolated!")
else:
    print("WARNING: Pitcher leakage detected!")

Total unique pitchers: 832
Train pitchers: 665
Test pitchers: 167

Train set: 482656 pitches from 482656 rows
Test set: 117758 pitches from 117758 rows

Pitcher leakage check: 0 overlapping pitchers
No pitcher leakage. Train and test are isolated!


## Preparing data for LSTM

For LSTM, we need sequences of pitches, not individual pitches. For each at-bat, we create a sequence of last N pitches as input.

In [13]:
def create_sequences_for_lstm(df_subset, feature_cols, target_col='pitch_type_encoded', seq_length=5):
    sequences = []
    targets = []
    
    # Group by at-bat (pitcher_batter_pair + game_date)
    df_sorted = df_subset.sort_values(['game_date', 'pitcher', 'batter', 'pitch_number'])
    
    for (game_date, pitcher, batter), group in df_sorted.groupby(['game_date', 'pitcher', 'batter']):
        if len(group) < 2:  # Need at least 2 pitches per AB
            continue
        
        # Get feature sequences and targets
        group_features = group[feature_cols].values
        group_targets = group[target_col].values
        
        # Create sliding windows: each window is [past N pitches] -> next pitch
        for i in range(1, len(group)):  # Start from 1 (need at least 1 prior pitch)
            # Input: features of the current pitch (which includes count, outs, etc.). This is the context for predicting the next pitch.
            sequences.append(group_features[i])
            targets.append(group_targets[i])
    
    return np.array(sequences), np.array(targets)
 
print("Creating LSTM sequences...")
X_train_lstm, y_train_lstm = create_sequences_for_lstm(df[train_idx], feature_cols)
X_test_lstm, y_test_lstm = create_sequences_for_lstm(df[test_idx], feature_cols)
 
print(f"LSTM Train sequences: {X_train_lstm.shape}")
print(f"LSTM Test sequences: {X_test_lstm.shape}")

Creating LSTM sequences...
LSTM Train sequences: (403984, 26)
LSTM Test sequences: (98161, 26)


## Exports for model training

In [14]:
# Save XGBoost data
np.save('X_train.npy', X_train.values)
np.save('X_test.npy', X_test.values)
np.save('y_train.npy', y_train.values)
np.save('y_test.npy', y_test.values)
 
print("Saved XGBoost data:")
print(f"  X_train.npy ({X_train.shape})")
print(f"  X_test.npy ({X_test.shape})")
print(f"  y_train.npy ({y_train.shape})")
print(f"  y_test.npy ({y_test.shape})")
 
# Save LSTM data
np.save('X_train_lstm.npy', X_train_lstm)
np.save('X_test_lstm.npy', X_test_lstm)
np.save('y_train_lstm.npy', y_train_lstm)
np.save('y_test_lstm.npy', y_test_lstm)
 
print("Saved LSTM data:")
print(f"  X_train_lstm.npy ({X_train_lstm.shape})")
print(f"  X_test_lstm.npy ({X_test_lstm.shape})")
print(f"  y_train_lstm.npy ({y_train_lstm.shape})")
print(f"  y_test_lstm.npy ({y_test_lstm.shape})")
 
# Save test pitcher IDs for later evaluation
test_pitcher_ids = df[test_idx]['pitcher'].values
np.save('test_pitcher_ids.npy', test_pitcher_ids)
print("Saved test_pitcher_ids.npy")
 
# Save feature names for model interpretation
with open('feature_names.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)
print("Saved feature_names.pkl")
 
# Save test dataframe info for analysis
test_df_info = df[test_idx][['game_date', 'pitcher', 'batter', 'pitch_type', 'pitch_type_encoded', 'balls', 'strikes']].copy()
test_df_info.to_csv('test_set_info.csv', index=False)
print("Saved test_set_info.csv")

Saved XGBoost data:
  X_train.npy ((482656, 26))
  X_test.npy ((117758, 26))
  y_train.npy ((482656,))
  y_test.npy ((117758,))
Saved LSTM data:
  X_train_lstm.npy ((403984, 26))
  X_test_lstm.npy ((98161, 26))
  y_train_lstm.npy ((403984,))
  y_test_lstm.npy ((98161,))
Saved test_pitcher_ids.npy
Saved feature_names.pkl
Saved test_set_info.csv
